In [ ]:
# !pip install dynabatch==0.1.7 --no-deps

Installation of Packages


In [ ]:
# transformers >= 5.5.0 for Gemma 4
!pip install transformers==5.5.0
!pip install bitsandbytes==0.49.2
!pip install -U transformers accelerate peft
!pip install dynabatch --no-deps

# Compare DynaBatch vs Normal Dataloaders

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig
import torch
from tqdm import tqdm
from datasets import load_dataset
from  dynabatch import build_dynamic_batch_dataloader, clear_gpu_memory, merge_outputs, split_batch
import torch
from torch.utils.data import Dataset, DataLoader, Sampler
from datetime import datetime
import numpy as np
import gc
import random

random.seed(21)

In [ ]:
# load data
ds = load_dataset("OpenAssistant/oasst1")
df = ds['train'].to_pandas()
wanted_langs = ["en", "de", "ru", "es"]
wanted_cols = ["text", "lang"]
df = df[df['lang'].isin(wanted_langs)]
df = df[wanted_cols]
device = torch.device("cuda")
df_es = df[df['lang'] == 'es']
df_en = df[df['lang'] == 'en']
df_ru = df[df['lang'] == 'ru']
df_de = df[df['lang'] == 'de']

# Color codes
GREEN = "\033[92m"
RED = "\033[91m"
RESET = "\033[0m"

In [ ]:
# BucketSampler for Bucket Loader (bucketing similar size)
class BucketSampler(Sampler):
    def __init__(self, data, batch_size, tokenizer):
        self.batch_size = batch_size
        self.lengths = np.array([len(text) for text in data])
        self.indices = np.argsort(self.lengths)

    def __iter__(self):
        batches = [
            self.indices[i : i + self.batch_size].tolist()
            for i in range(0, len(self.indices), self.batch_size)
        ]
        for batch in batches:
            yield from batch

    def __len__(self):
        return len(self.lengths)

## 1. Model: facebook/m2m100_418M

In [ ]:
model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M", torch_dtype=torch.float16).to(device)
tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")
tokenizer.src_lang = "es"
max_length = 512
samples = 2000
texts_data = df_es["text"].sample(samples, random_state=21).tolist()

In [ ]:
def inference_m2m(custom_dataloader):
  start_time = datetime.now()
  with torch.inference_mode():
    for i, batch in tqdm(enumerate(custom_dataloader), total=len(custom_dataloader)):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        generated_tokens = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            forced_bos_token_id=tokenizer.get_lang_id("en")
        )
  time_taken = (datetime.now() - start_time).total_seconds()
  del input_ids, attention_mask, generated_tokens
  gc.collect()
  torch.cuda.empty_cache()
  return time_taken


**1.1 Normal Dataloader**

In [ ]:
M2M100_BATCH_SIZE = 64

# load dataloader
normal_loader = DataLoader(
    texts_data,
    batch_size=M2M100_BATCH_SIZE,
    collate_fn=lambda batch: tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=max_length)
)
print("*"*20, "Using Normal loader", "*"*20)
m2m100_normal_dataloader_time = inference_m2m(normal_loader)
print(F"M2M100-418M - Normal Dataloader Time: {m2m100_normal_dataloader_time} seconds")


**1.2 Bucket Sampler Dataloader**

In [ ]:
M2M100_BATCH_SIZE = 64

# load dataloader
bucket_sampler = BucketSampler(texts_data, M2M100_BATCH_SIZE, tokenizer)
bucket_loader = DataLoader(
    texts_data,
    batch_size=M2M100_BATCH_SIZE,
    sampler=bucket_sampler,
    collate_fn=lambda batch: tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=max_length))

print("*"*20, "Using Bucket loader", "*"*20)
m2m100_bucket_sampler_time = inference_m2m(bucket_loader)
print(F"M2M100-418M - Bucket Dataloader Time: {m2m100_bucket_sampler_time} seconds")

**1.3 DynaBatch dataloader**

In [ ]:
M2M100_BATCH_SIZE = 64

# load dataloader
dynbatch_loader = build_dynamic_batch_dataloader(texts_data, tokenizer=tokenizer, batch_size=M2M100_BATCH_SIZE, max_input_token_length=max_length, threshold=0.025)

print("*"*20, "Using Dynabatch loader", "*"*20)
m2m100_dynabatch_time = inference_m2m(dynbatch_loader)
print(f"M2M100-418M - DynaBatch Time: {m2m100_dynabatch_time} seconds")

**1.4 Compare Time taken**

In [ ]:
times = m2m100_normal_dataloader_time / m2m100_dynabatch_time
color = GREEN if times > 1 else RED
print(f"Dynabatch is {color}{times:.2f}{RESET} times faster than Normal dataloader for M2M100-418M")

times = m2m100_bucket_sampler_time / m2m100_dynabatch_time
color = GREEN if times > 1 else RED
print(f"Dynabatch is {color}{times:.2f}{RESET} times faster than Bucket dataloader for M2M100-418M")
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

## 2 Model: google/gemma-4-E2B-it

In [ ]:
MODEL_ID = "google/gemma-4-E2B-it"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"
processor.tokenizer.pad_token = processor.tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto"
)
model = torch.compile(model)

In [ ]:
max_length = 128
data_sample_size = 1250


def create_every_sizes(data):
    """
    since T4 is super slow to run Gemma 4, max_length is set to 128
    but this would lead having most text length of 124. So varying it to keep
    it more real world data
    """
    n_samples = data_sample_size // 5
    length_config = [
        {"min_length": 1, "max_length": 10, "samples": n_samples},
        {"min_length": 10, "max_length": 25, "samples": n_samples},
        {"min_length": 25, "max_length": 50, "samples": n_samples},
        {"min_length": 50, "max_length": 80, "samples": n_samples},
        {"min_length": 80, "max_length": max_length, "samples": n_samples},
    ]
    lenght_config = []
    new_data = []
    for config in length_config:
        for _ in range(config["samples"]):
            words = []
            while True:
                words.extend(random.choice(data).split())
                if len(words) >= config["min_length"]:
                    words = words[:config["max_length"]]
                    break
            words = " ".join(words)
            new_data.append(words)
    return new_data


def format_translation_batch(texts):
    formatted = []


    for t in texts:
        messages = [
            {"role": "user", "content": f"Translate German to English: {t}"}
        ]
        formatted.append(processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
    return formatted


def inference_gemma(custom_dataloader):
  start_time = datetime.now()
  model.eval()
  with torch.inference_mode():
    for i, batch in tqdm(enumerate(custom_dataloader), total=len(custom_dataloader)):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        generated_tokens = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            do_sample=False,
            pad_token_id=processor.tokenizer.pad_token_id,
            max_new_tokens=max_length,
        )

  time_taken = (datetime.now() - start_time).total_seconds()
  del input_ids, attention_mask, generated_tokens
  gc.collect()
  torch.cuda.empty_cache()
  return time_taken

df_de_sample = df_de.sample(data_sample_size, random_state=21)['text'].to_list()
texts_data = create_every_sizes(df_de_sample)
texts_data = format_translation_batch(texts_data)

In [ ]:
texts_data[2]

**2.1 Normal Dataloader**

In [ ]:
GEMMA_BATCH_SIZE = 96
# load dataloader
normal_loader = DataLoader(
    texts_data,
    batch_size=GEMMA_BATCH_SIZE,
    collate_fn=lambda batch: processor(text=batch, padding=True, truncation=True, return_tensors="pt", max_length=max_length)
)

print("*"*20, "Using Normal loader", "*"*20)
gemma_normal_dataloader_time = inference_gemma(normal_loader)
print(F"Gemma 4 - Normal Dataloader Time: {gemma_normal_dataloader_time} seconds")

**2.2 Bucket Sampler Dataloader**

In [ ]:
GEMMA_BATCH_SIZE = 96

bucket_sampler = BucketSampler(texts_data, GEMMA_BATCH_SIZE, processor)
bucket_loader = DataLoader(
    texts_data,
    batch_size=GEMMA_BATCH_SIZE,
    sampler=bucket_sampler,
    collate_fn=lambda batch: processor(text=batch, padding=True, truncation=True, return_tensors="pt", max_length=max_length))

print("*"*20, "Using Bucket loader", "*"*20)
gemma_bucket_sampler_time = inference_gemma(bucket_loader)
print(F"Gemma 4 - Bucket Dataloader Time: {gemma_bucket_sampler_time} seconds")

**2.3 DynaBatch dataloader**

In [ ]:
GEMMA_BATCH_SIZE = 96

dynbatch_loader = build_dynamic_batch_dataloader(texts_data, tokenizer=processor, batch_size=GEMMA_BATCH_SIZE, max_input_token_length=max_length, threshold=0.025)

print("*"*20, "Using Dynabatch loader", "*"*20)
gemma_dynabatch_time = inference_gemma(dynbatch_loader)
print(f"Gemma 4 - DynaBatch Time: {gemma_dynabatch_time} seconds")

**2.4 Compare Time taken**

In [ ]:
times = gemma_normal_dataloader_time / gemma_dynabatch_time
color = GREEN if times > 1 else RED
print(f"Dynabatch is {color}{times:.2f}{RESET} times faster than Normal dataloader for Gemma 4")

times = gemma_bucket_sampler_time / gemma_dynabatch_time
color = GREEN if times > 1 else RED
print(f"Dynabatch is {color}{times:.2f}{RESET} times faster than Bucket dataloader for Gemma 4")
del model, processor
gc.collect()
torch.cuda.empty_cache()

##3. Model: facebook/nllb-200-distilled-600M

In [ ]:
model_name = "facebook/nllb-200-distilled-600M"
source_lang_code = "eng_Latn"
target_lang_code = "rus_Cyrl"
tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang=source_lang_code, tgt_lang=target_lang_code)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16
).to(device)
model = torch.compile(model)

samples = 10000
texts_data = df_en["text"].sample(samples, random_state=21).tolist()
max_length = 512

In [ ]:
def inference_nllb(custom_dataloader):
    start_time = datetime.now()
    with torch.inference_mode():
      for i, batch in tqdm(enumerate(custom_dataloader), total=len(custom_dataloader)):
          input_ids = batch["input_ids"].to(device)
          attention_mask = batch["attention_mask"].to(device)
          generated_tokens = model.generate(
              input_ids=input_ids,
              attention_mask=attention_mask,
          )
    time_taken = (datetime.now() - start_time).total_seconds()
    del input_ids, attention_mask, generated_tokens
    gc.collect()
    torch.cuda.empty_cache()
    return time_taken


**3.1 Normal Dataloader**

In [ ]:
NLLB_BATCH_SIZE = 304
# load dataloader
normal_loader = DataLoader(
    texts_data,
    batch_size=NLLB_BATCH_SIZE,
    collate_fn=lambda batch: tokenizer(text=batch, padding=True, truncation=True, return_tensors="pt", max_length=max_length)
)

print("*"*20, "Using Normal loader", "*"*20)
nllb_normal_dataloader_time = inference_nllb(normal_loader)
print(F"NLLB - Normal Dataloader Time: {nllb_normal_dataloader_time} seconds")

**3.2 Bucket Dataloader**

In [ ]:
NLLB_BATCH_SIZE = 304

bucket_sampler = BucketSampler(texts_data, NLLB_BATCH_SIZE, tokenizer)
bucket_loader = DataLoader(
    texts_data,
    batch_size=NLLB_BATCH_SIZE,
    sampler=bucket_sampler,
    collate_fn=lambda batch: tokenizer(text=batch, padding=True, truncation=True, return_tensors="pt", max_length=max_length))

print("*"*20, "Using Bucket loader", "*"*20)
nllb_bucket_sampler_time = inference_nllb(bucket_loader)
print(F"NLLB - Bucket Dataloader Time: {nllb_bucket_sampler_time} seconds")

**3.3 Dynabatch**

In [ ]:
from dynabatch import generate_with_oom_fallback

def inference_nllb_with_OOM_handle(custom_dataloader, min_batch_size):
    """
    One limitation of dynabatch is that sometimes the dynabatch model incorrectly
    predicts bigger batch size, leading to GPU OOM error. To workaround this
    without interrupting the flow of inference, we use generate_with_oom_fallback
    which splits the batch into smaller chunks and merges results back.
    """
    failed_times = 0
    start_time = datetime.now()
    with torch.inference_mode():
        for i, batch in tqdm(enumerate(custom_dataloader), total=len(custom_dataloader)):
            generated_tokens, did_fallback = generate_with_oom_fallback(
                model, batch, min_batch_size=min_batch_size, device=device
            )
            if did_fallback:
                failed_times += 1

    print("Total Failed: ", failed_times)
    time_taken = (datetime.now() - start_time).total_seconds()
    del generated_tokens
    gc.collect()
    torch.cuda.empty_cache()
    return time_taken

In [ ]:
NLLB_BATCH_SIZE = 304

dynbatch_loader = build_dynamic_batch_dataloader(
    texts_data,
    tokenizer=tokenizer,
    batch_size=NLLB_BATCH_SIZE,
    max_input_token_length=max_length,
    threshold=0.025,
    max_batch_size=NLLB_BATCH_SIZE
)

print("*"*20, "Using Dynabatch loader", "*"*20)
nllb_dynabatch_time = inference_nllb_with_OOM_handle(dynbatch_loader, NLLB_BATCH_SIZE)
print(f"NLLB - DynaBatch Time: {nllb_dynabatch_time} seconds")

**3.4 Compare Time taken**

In [ ]:
times = nllb_normal_dataloader_time / nllb_dynabatch_time
color = GREEN if times > 1 else RED
print(f"Dynabatch is {color}{times:.2f}{RESET} times faster than Normal dataloader for NLLB")

times = nllb_bucket_sampler_time / nllb_dynabatch_time
color = GREEN if times > 1 else RED
print(f"Dynabatch is {color}{times:.2f}{RESET} times faster than Bucket dataloader for NLLB")
# del model, processor
# gc.collect()
# torch.cuda.empty_cache()